# Download NYCCAS data (download_NYCCAS.ipynb)

**Purpose:** Download monthly NYCCAS CSV files (October 2019 onward), save them into `data_csv/NYCCAS/`, merge them into a single dataset, and generate site metadata and seasonal/annual aggregates for downstream analysis.

**Data sources:**
- NYCCAS GitHub raw CSV archive for monthly pollution observations
- NYCCAS station metadata from `station-new.csv`

**Primary outputs:**
- `data_csv/NYCCAS/` — monthly raw CSV files
- `data_csv/NYCCAS_2019-2026.csv` — merged monthly observations
- `data_csv/NYCCAS/site-info.csv` — station metadata
- `data_csv/NYCCAS-Annual.csv`
- `data_csv/NYCCAS-Winter.csv`
- `data_csv/NYCCAS-Spring.csv`
- `data_csv/NYCCAS-Summer.csv`
- `data_csv/NYCCAS-Fall.csv`
- `data_csv/NYCCAS-Winter-Borough.csv`
- `data_csv/NYCCAS-Spring-Borough.csv`
- `data_csv/NYCCAS-Summer-Borough.csv`
- `data_csv/NYCCAS-Fall-Borough.csv`
- `data_csv/NYCCAS-Seasonal-Borough.csv`
- `data_csv/NYCCAS-Yearly-Borough.csv`

**Dependencies:**
- Python 3.8+
- `urllib`, `ssl`, `certifi`, `pandas`, `numpy`

**How to run:**
1. Open this notebook in Jupyter.
2. Ensure network connectivity so the GitHub raw CSV files and station metadata can be downloaded.
3. Execute the notebook cells sequentially.
4. If using a virtual environment, activate it first and install requirements from `requirements.txt` or `environment.yml`.

**Notes & reproducibility:**
- The notebook downloads monthly data files and concatenates them into one master CSV.
- It also downloads station metadata and merges location names into the merged dataset.
- If the station metadata column names change, adjust the `name_col` lookup accordingly.
- Keep network and GitHub access available while running this notebook.

**Author / Maintainer:** Mayank Anand — mayank.anand3007@gmail.com

In [1]:
# Consolidated imports
import os
import urllib.request
import ssl
import certifi
import pandas as pd
import numpy as np

# SSL context for urllib requests using certifi CA bundle
ssl_context = ssl.create_default_context(cafile=certifi.where())

In [2]:
# Ensure output directory for monthly NYCCAS CSVs exists
out_dir = "data_csv/NYCCAS"
os.makedirs(out_dir, exist_ok=True)
print(f"Ensured directory exists: {out_dir}")

# iterate from Oct 2019 through Feb 2026 and save each month’s file
for year in range(2019, 2027):
    for month in range(1, 13):
        # stop when we have written February…2026
        if year == 2026 and month > 2:
            break
        # skip months before October 2019
        if year == 2019 and month < 10:
            continue

        url = f"https://raw.githubusercontent.com/nychealth/nyccas-data/refs/heads/main/hist/csv/{year}/{month}.csv"
        try:
            resp = urllib.request.urlopen(url, context=ssl_context)
            df = pd.read_csv(resp)                  # read month’s data
            df.to_csv(f"data_csv/NYCCAS/{year}_{month:02d}.csv", index=False)  # save to disk
            print(f"saved {year}-{month:02d}.csv")
        except Exception as exc:
            print(f"failed to fetch {year}-{month:02d}: {exc}")

Ensured directory exists: data_csv/NYCCAS
saved 2019-10.csv
saved 2019-11.csv
saved 2019-12.csv
saved 2020-01.csv
saved 2020-02.csv
saved 2020-03.csv
saved 2020-04.csv
saved 2020-05.csv
saved 2020-06.csv
saved 2020-07.csv
saved 2020-08.csv
saved 2020-09.csv
saved 2020-10.csv
saved 2020-11.csv
saved 2020-12.csv
saved 2021-01.csv
saved 2021-02.csv
saved 2021-03.csv
saved 2021-04.csv
saved 2021-05.csv
saved 2021-06.csv
saved 2021-07.csv
saved 2021-08.csv
saved 2021-09.csv
saved 2021-10.csv
saved 2021-11.csv
saved 2021-12.csv
saved 2022-01.csv
saved 2022-02.csv
saved 2022-03.csv
saved 2022-04.csv
saved 2022-05.csv
saved 2022-06.csv
saved 2022-07.csv
saved 2022-08.csv
saved 2022-09.csv
saved 2022-10.csv
saved 2022-11.csv
saved 2022-12.csv
saved 2023-01.csv
saved 2023-02.csv
saved 2023-03.csv
saved 2023-04.csv
saved 2023-05.csv
saved 2023-06.csv
saved 2023-07.csv
saved 2023-08.csv
saved 2023-09.csv
saved 2023-10.csv
saved 2023-11.csv
saved 2023-12.csv
saved 2024-01.csv
saved 2024-02.csv
save

In [3]:
out_path = "data_csv/NYCCAS/site-info.csv"
os.makedirs(os.path.dirname(out_path), exist_ok=True)

station_raw = "https://raw.githubusercontent.com/nychealth/nyccas-data/refs/heads/main/portal/station-new.csv"
try:
    with urllib.request.urlopen(station_raw, context=ssl_context) as resp:
        data = resp.read()
    with open(out_path, "wb") as f:
        f.write(data)
    print(f"Saved station-new.csv → {out_path}")
except Exception as exc:
    print(f"Failed to fetch station-new.csv from raw URL: {exc}")

Saved station-new.csv → data_csv/NYCCAS/site-info.csv


In [4]:
base_dir = "data_csv/NYCCAS"          # adjust if the path is elsewhere
monthly_frames = []

for root, dirs, files in os.walk(base_dir):
	for fn in files:
		if not fn.endswith(".csv") or fn == "site-info.csv":
			continue
		path = os.path.join(root, fn)
		stem = os.path.splitext(fn)[0]    # e.g. "2020_01"
		parts = stem.split("_")
		if len(parts) < 2:
			# not a year_month file
			continue
		try:
			yr = int(parts[0])
			mo = int(parts[1])
		except ValueError:
			continue
		df_month = pd.read_csv(path)
		monthly_frames.append(df_month)

# merge all the monthly frames into one
merged_monthly = pd.concat(monthly_frames, ignore_index=True)

# save result
merged_monthly.to_csv("data_csv/NYCCAS_2019-2026.csv", index=False)

print("merged", len(monthly_frames), "files;", merged_monthly.shape, "rows in output")

merged 77 files; (374373, 4) rows in output


In [5]:
# read the site-info file that lives alongside the monthly csvs
site_info = pd.read_csv(os.path.join(base_dir, "site-info.csv"))
print("columns:", site_info.columns.tolist())

# typically the file has a SiteID column and something like SiteName;
# adjust the name below if it is different in your copy
name_col = "Location" if "Location" in site_info.columns else site_info.columns[1]

# merge the name into the big frame and give it a consistent column name
merged_with_siteinfo = merged_monthly.merge(site_info[["SiteID", name_col]],
                      on="SiteID",
                      how="left")
merged_with_siteinfo = merged_with_siteinfo.rename(columns={name_col: "SiteName"})
merged_full = merged_with_siteinfo

# write the augmented table back out (overwrite or write a new file)
merged_full.to_csv("data_csv/NYCCAS_2019-2026.csv", index=False)

columns: ['SiteID', 'Location', 'loc_col', 'Latitude', 'Longitude']


In [6]:
# Work from the merged in-memory frame so we avoid a redundant read/write cycle.
df = merged_full.copy()

# convert the timestamp column and extract month/year directly
# (ObservationTimeUTC has format yyyy-mm-dd hh:mm:ss...)
df["ObservationTimeUTC"] = pd.to_datetime(df["ObservationTimeUTC"])
df["month"] = df["ObservationTimeUTC"].dt.month
df["Year"] = df["ObservationTimeUTC"].dt.year
df = df.drop(columns=["year"], errors="ignore")

In [7]:
# Example dataframe (replace with your actual df)
# df = pd.read_csv("your_file.csv")

# Mapping dictionary
site_to_borough = {
    'Midtown West': 'Manhattan',
    'Queensboro Bridge': 'Queens',
    'Williamsburg Bridge': 'Brooklyn',
    'Queens College': 'Queens',
    'Cross Bronx Expy': 'Bronx',
    'Manhattan Bridge': 'Manhattan',
    'Broadway/35th St': 'Manhattan',
    'Mott Haven': 'Bronx',
    'FDR': 'Manhattan',  # FDR Drive
    'Van Wyck': 'Queens',  # Van Wyck Expressway
    'SI Expwy': 'Staten Island',
    'BQE': 'Brooklyn',  # Brooklyn-Queens Expressway (choose dominant borough)
    'Hamilton Bridge': 'Bronx',
    "Hunt's Point": 'Bronx',
    'Glendale': 'Queens'
}

# Map boroughs
df['Borough'] = df['SiteName'].map(site_to_borough)

# Drop rows where Borough is NaN
df = df.dropna(subset=['Borough'])

In [8]:
def save_seasonal_mean(frame, months, group_cols, output_path, *, season_name=None, roll_year_months=()):
    seasonal_frame = frame[frame["month"].isin(months)].copy()
    if roll_year_months:
        seasonal_frame["Year"] = np.where(
            seasonal_frame["month"].isin(roll_year_months),
            seasonal_frame["Year"] + 1,
            seasonal_frame["Year"],
        )
    aggregated = seasonal_frame.groupby(list(group_cols), as_index=False)["Value"].mean()
    if season_name is not None:
        aggregated["Season"] = season_name
    aggregated.to_csv(output_path, index=False)
    print("saved", aggregated.shape[0], "rows to", output_path)
    return aggregated

In [9]:
winter_yearly = save_seasonal_mean(df, [12, 1, 2], ["Year"], "data_csv/NYCCAS-Winter-yearly.csv", roll_year_months=(12,))
summer_yearly = save_seasonal_mean(df, [6, 7, 8], ["Year"], "data_csv/NYCCAS-Summer-yearly.csv")
spring_yearly = save_seasonal_mean(df, [3, 4, 5], ["Year"], "data_csv/NYCCAS-Spring-yearly.csv")
fall_yearly = save_seasonal_mean(df, [9, 10, 11], ["Year"], "data_csv/NYCCAS-Fall-yearly.csv")

winter_by_site = save_seasonal_mean(df, [12, 1, 2], ["SiteName", "Year"], "data_csv/NYCCAS-Winter.csv", roll_year_months=(12,))
summer_by_site = save_seasonal_mean(df, [6, 7, 8], ["SiteName", "Year"], "data_csv/NYCCAS-Summer.csv")
spring_by_site = save_seasonal_mean(df, [3, 4, 5], ["SiteName", "Year"], "data_csv/NYCCAS-Spring.csv")
fall_by_site = save_seasonal_mean(df, [9, 10, 11], ["SiteName", "Year"], "data_csv/NYCCAS-Fall.csv")

annual_by_site = df.groupby(["SiteName", "Year"], as_index=False)["Value"].mean()
annual_by_site.to_csv("data_csv/NYCCAS-Annual.csv", index=False)
print("saved", annual_by_site.shape[0], "rows to data_csv/NYCCAS-Annual.csv")

winter_by_borough = save_seasonal_mean(df, [11, 12, 1], ["Borough", "Year"], "data_csv/NYCCAS-Winter-Borough.csv", roll_year_months=(11, 12), season_name="Winter")
summer_by_borough = save_seasonal_mean(df, [6, 7, 8], ["Borough", "Year"], "data_csv/NYCCAS-Summer-Borough.csv", season_name="Summer")
spring_by_borough = save_seasonal_mean(df, [3, 4, 5], ["Borough", "Year"], "data_csv/NYCCAS-Spring-Borough.csv", season_name="Spring")
fall_by_borough = save_seasonal_mean(df, [9, 10, 11], ["Borough", "Year"], "data_csv/NYCCAS-Fall-Borough.csv", season_name="Fall")

combined_seasonal = pd.concat([winter_by_borough, summer_by_borough, spring_by_borough, fall_by_borough], ignore_index=True)
combined_seasonal.to_csv("data_csv/NYCCAS-Seasonal-Borough.csv", index=False)
print("saved", combined_seasonal.shape[0], "rows to data_csv/NYCCAS-Seasonal-Borough.csv")

yearly_by_borough = df.groupby(["Borough", "Year"], as_index=False)["Value"].mean()
yearly_by_borough.to_csv("data_csv/NYCCAS-Yearly-Borough.csv", index=False)
print("saved", yearly_by_borough.shape[0], "rows to data_csv/NYCCAS-Yearly-Borough.csv")

saved 7 rows to data_csv/NYCCAS-Winter-yearly.csv
saved 6 rows to data_csv/NYCCAS-Summer-yearly.csv
saved 6 rows to data_csv/NYCCAS-Spring-yearly.csv
saved 7 rows to data_csv/NYCCAS-Fall-yearly.csv
saved 57 rows to data_csv/NYCCAS-Winter.csv
saved 49 rows to data_csv/NYCCAS-Summer.csv
saved 52 rows to data_csv/NYCCAS-Spring.csv
saved 51 rows to data_csv/NYCCAS-Fall.csv
saved 77 rows to data_csv/NYCCAS-Annual.csv
saved 23 rows to data_csv/NYCCAS-Winter-Borough.csv
saved 22 rows to data_csv/NYCCAS-Summer-Borough.csv
saved 24 rows to data_csv/NYCCAS-Spring-Borough.csv
saved 22 rows to data_csv/NYCCAS-Fall-Borough.csv
saved 91 rows to data_csv/NYCCAS-Seasonal-Borough.csv
saved 33 rows to data_csv/NYCCAS-Yearly-Borough.csv
